# Sprint 12 - Consumo de datos desde una API REST

En este notebook trabajo con APIs REST utilizando Python.  
El objetivo principal es practicar cómo hacer peticiones HTTP, revisar los códigos de estado, interpretar respuestas en formato JSON y convertir los datos obtenidos en DataFrames de pandas.

Durante el ejercicio se utilizan tres tipos de APIs:

- JSONPlaceholder, como API de laboratorio para practicar métodos HTTP.
- Una API pública real, para consultar datos externos mediante peticiones GET.
- Open Data Barcelona, para buscar datasets públicos y guardar los resultados en un archivo CSV.

La práctica está organizada en tres niveles, siguiendo el enunciado del Sprint 12.

## Librerías utilizadas

Para resolver este sprint utilizo principalmente las siguientes librerías:

- `requests`: para hacer peticiones HTTP a las APIs REST y revisar las respuestas recibidas.
- `pandas`: para convertir los datos obtenidos en DataFrames y poder analizarlos de forma más clara.
- `json`: para visualizar algunas respuestas JSON de una manera más ordenada y fácil de leer.

Estas librerías me permiten conectar con APIs externas, comprobar los códigos de estado, interpretar respuestas en formato JSON y transformar los datos en una estructura tabular para trabajar con ellos.

In [2]:
import requests
import pandas as pd
import json

Las librerías se han importado correctamente. A partir de aquí ya puedo empezar a realizar peticiones HTTP y trabajar con los datos obtenidos.

## Nivel 1 - Exploración básica con JSONPlaceholder

En este primer nivel utilizo la API pública JSONPlaceholder.  
Esta API es una herramienta de prueba que permite practicar peticiones HTTP sin modificar datos reales.

En esta parte voy a trabajar con diferentes métodos HTTP:

- `GET`, para consultar datos.
- `POST`, para simular la creación de un nuevo recurso.
- `PATCH`, para simular una modificación parcial.
- `DELETE`, para simular la eliminación de un recurso.

Aunque las peticiones `POST`, `PATCH` y `DELETE` devuelven una respuesta, JSONPlaceholder no guarda los cambios de forma real en el servidor. Es una API pensada para practicar.

### 1.1 y 1.2 Consulta de recursos con GET y revisión de resultados

En este apartado hago peticiones `GET` a tres recursos de la API JSONPlaceholder: `posts`, `users` y `todos`.

El objetivo de esta parte es doble. Primero, consultar cada endpoint para obtener los datos disponibles. Después, revisar dos elementos importantes de cada respuesta:

- El código de estado HTTP, para comprobar si la petición se ha realizado correctamente.
- La cantidad total de registros recibidos, para saber cuántos elementos devuelve cada recurso.

En este caso utilizo un bucle `for` para evitar repetir el mismo código tres veces. Así puedo consultar varios endpoints de una forma más ordenada y clara.

In [3]:
base_url = "https://jsonplaceholder.typicode.com"

resources = ["posts", "users", "todos"]

for resource in resources:
    url = f"{base_url}/{resource}"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        print(f"Recurso: {resource}")
        print(f"Código de estado: {response.status_code}")
        print(f"Cantidad total de registros: {len(data)}")
        print("-" * 40)
    else:
        print(f"Error al consultar el recurso: {resource}")
        print(f"Código de estado: {response.status_code}")
        print("-" * 40)

Recurso: posts
Código de estado: 200
Cantidad total de registros: 100
----------------------------------------
Recurso: users
Código de estado: 200
Cantidad total de registros: 10
----------------------------------------
Recurso: todos
Código de estado: 200
Cantidad total de registros: 200
----------------------------------------


En esta primera parte he consultado los recursos `posts`, `users` y `todos` de JSONPlaceholder mediante peticiones `GET`.

Las tres peticiones han devuelto un código de estado `200`, lo que indica que se han realizado correctamente. Además, he podido comprobar la cantidad total de registros disponibles en cada endpoint.

Esta comprobación es importante porque antes de transformar o analizar datos de una API, primero hay que verificar que la conexión funciona y que la respuesta recibida es válida.

### 1.3 Prueba de error 404

En este apartado hago una petición `GET` a una publicación que no existe dentro de JSONPlaceholder.

El objetivo es comprobar cómo responde la API cuando se solicita un recurso inexistente.  
En este caso espero recibir un código de estado `404`, que indica que el recurso no ha sido encontrado.

In [4]:
url_not_found = f"{base_url}/posts/999999"

response_not_found = requests.get(url_not_found)

print(f"URL consultada: {url_not_found}")
print(f"Código de estado recibido: {response_not_found.status_code}")

URL consultada: https://jsonplaceholder.typicode.com/posts/999999
Código de estado recibido: 404


La petición a una publicación inexistente ha devuelto el código de estado `404`.

Este resultado indica que la API está disponible, pero el recurso solicitado no existe.  
Esta prueba es útil para entender cómo interpretar errores HTTP y cómo diferenciar una petición correcta de una petición que no encuentra el recurso solicitado.

### 1.4 Creación de una publicación ficticia con POST

En este apartado hago una petición `POST` para simular la creación de una nueva publicación en JSONPlaceholder.

Para crear esta publicación ficticia envío tres campos principales:

- `title`: el título de la publicación.
- `body`: el contenido del mensaje.
- `userId`: el identificador del usuario asociado a la publicación.

JSONPlaceholder devuelve una respuesta en formato JSON, pero no guarda realmente los datos en el servidor. Esta API está pensada para practicar peticiones HTTP.

In [5]:
url_posts = f"{base_url}/posts"

new_post = {
    "title": "Nueva publicación de prueba",
    "body": "Este es el contenido de una publicación ficticia creada para practicar una petición POST.",
    "userId": 1
}

response_post = requests.post(url_posts, json=new_post)

print(f"Código de estado: {response_post.status_code}")
print("Respuesta JSON:")
print(json.dumps(response_post.json(), indent=4, ensure_ascii=False))

Código de estado: 201
Respuesta JSON:
{
    "title": "Nueva publicación de prueba",
    "body": "Este es el contenido de una publicación ficticia creada para practicar una petición POST.",
    "userId": 1,
    "id": 101
}


La petición `POST` ha devuelto un código de estado `201`, que indica que la creación ficticia del recurso se ha realizado correctamente.

La respuesta JSON incluye los campos enviados en la petición y también un nuevo campo `id`, generado por la API.  
Aunque JSONPlaceholder devuelve una respuesta correcta, los datos no se guardan de forma permanente porque se trata de una API de prueba.

### 1.5 Modificación parcial de una publicación con PATCH

En este apartado hago una petición `PATCH` para modificar parcialmente una publicación existente en JSONPlaceholder.

A diferencia de `POST`, que se utiliza para crear un nuevo recurso, `PATCH` se utiliza para actualizar solo algunos campos de un recurso ya existente.  
En este caso modifico únicamente el título de una publicación.

In [6]:
url_patch = f"{base_url}/posts/1"

updated_data = {
    "title": "Título actualizado con PATCH"
}

response_patch = requests.patch(url_patch, json=updated_data)

print(f"Código de estado: {response_patch.status_code}")
print("Respuesta JSON:")
print(json.dumps(response_patch.json(), indent=4, ensure_ascii=False))

Código de estado: 200
Respuesta JSON:
{
    "userId": 1,
    "id": 1,
    "title": "Título actualizado con PATCH",
    "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
}


### 1.6 Eliminación de una publicación con DELETE

En este apartado hago una petición `DELETE` sobre una publicación existente de JSONPlaceholder.

El método `DELETE` se utiliza para solicitar la eliminación de un recurso. En este caso elimino de forma ficticia la publicación con id `1`.

Como JSONPlaceholder es una API de prueba, la petición devuelve una respuesta correcta, pero el recurso no se elimina realmente del servidor.

In [7]:
url_delete = f"{base_url}/posts/1"

response_delete = requests.delete(url_delete)

print(f"Código de estado: {response_delete.status_code}")
print("Respuesta JSON:")
print(response_delete.json())

Código de estado: 200
Respuesta JSON:
{}


La petición `DELETE` ha devuelto un código de estado `200`, lo que indica que la solicitud se ha procesado correctamente.

La respuesta JSON aparece vacía (`{}`), algo habitual en este tipo de operaciones cuando no es necesario devolver información adicional.

En este caso he imprimido directamente `response_delete.json()` porque la respuesta es muy simple. En las peticiones `POST` y `PATCH` he utilizado `json.dumps()` para mostrar respuestas más completas de forma ordenada, pero aquí no era necesario añadir formato extra.

Mantener el código simple también es una buena práctica: si una línea adicional no mejora la claridad, el resultado o el mantenimiento del código, es mejor evitarla.

## Nivel 2 - Interacción con una API pública real

En este segundo nivel trabajo con una API pública real para practicar la consulta de datos externos mediante peticiones `GET`.

La API elegida es **REST Countries API**, que permite obtener información sobre países, regiones, capitales, población, idiomas, monedas y otros datos generales.

He elegido esta API porque no requiere autenticación, devuelve respuestas en formato JSON y ofrece datos estructurados que se pueden convertir fácilmente en un DataFrame de pandas.

### 2.1 y 2.2 API elegida y revisión de la documentación

Para este nivel he revisado el repositorio de Public APIs con el objetivo de elegir una API pública que permitiera hacer peticiones `GET`, devolviera datos en formato JSON y no requiriera autenticación.

Primero he considerado APIs más relacionadas con sostenibilidad y huella de carbono, como CO2 Offset, porque este tema puede estar conectado con proyectos de logística sostenible como Kamport. Sin embargo, para este ejercicio he decidido no utilizarla porque no encontré una documentación suficientemente clara con endpoints sencillos y estables para hacer una consulta `GET` y transformar la respuesta en un DataFrame.

Por este motivo, finalmente he elegido **REST Countries API**. Esta API permite consultar información general sobre países de todo el mundo, no requiere autenticación, devuelve respuestas en formato JSON y ofrece datos estructurados que se pueden convertir fácilmente en un DataFrame de pandas.

Algunos endpoints útiles de REST Countries API son:

- `/v3.1/all`: devuelve información de todos los países.
- `/v3.1/name/{name}`: permite buscar un país por su nombre.
- `/v3.1/region/{region}`: permite filtrar países por región, por ejemplo `europe`, `asia` o `americas`.

También ofrece parámetros interesantes, como:

- `fields`: permite limitar los campos que devuelve la API y recibir solo la información necesaria.
- `fullText`: permite hacer una búsqueda más exacta por nombre cuando se utiliza el endpoint de búsqueda por país.

La respuesta se recibe en formato JSON, que es adecuado para este ejercicio porque se puede trabajar fácilmente con Python y convertir posteriormente en un DataFrame.

### 2.3 Petición GET sencilla

En este apartado hago una petición `GET` a REST Countries API para obtener información sobre países de Europa.

Utilizo el endpoint `/v3.1/region/europe`, que permite filtrar los países por región. Además, uso el parámetro `fields` para pedir solo algunos campos concretos y evitar recibir información innecesaria.

Los campos seleccionados son:

- `name`: nombre del país.
- `capital`: capital o capitales del país.
- `region`: región principal.
- `population`: población.
- `languages`: idiomas oficiales.
- `currencies`: monedas utilizadas.

Después de recibir la respuesta, reviso el código de estado y muestro algunos campos de varios países para comprobar que los datos llegan correctamente en formato JSON.

In [9]:
countries_url = "https://restcountries.com/v3.1/region/europe"

params = {
    "fields": "name,capital,region,population,languages,currencies"
}

response_countries = requests.get(countries_url, params=params)

print(f"Código de estado: {response_countries.status_code}")

if response_countries.status_code == 200:
    countries_data = response_countries.json()
    
    print(f"Cantidad de países recibidos: {len(countries_data)}")
    print("\nEjemplos de países recibidos:\n")
    
    for country in countries_data[:5]:
        name = country.get("name", {}).get("common", "No disponible")
        
        capital_list = country.get("capital", [])
        capital = ", ".join(capital_list) if capital_list else "No disponible"
        
        region = country.get("region", "No disponible")
        population = country.get("population", "No disponible")
        
        print(f"País: {name}")
        print(f"Capital: {capital}")
        print(f"Región: {region}")
        print(f"Población: {population}")
        print("-" * 40)
else:
    print("No se ha podido obtener la información de la API.")

Código de estado: 200
Cantidad de países recibidos: 53

Ejemplos de países recibidos:

País: Slovenia
Capital: Ljubljana
Región: Europe
Población: 2130638
----------------------------------------
País: Sweden
Capital: Stockholm
Región: Europe
Población: 10605098
----------------------------------------
País: Switzerland
Capital: Bern
Región: Europe
Población: 9082848
----------------------------------------
País: Guernsey
Capital: St. Peter Port
Región: Europe
Población: 64781
----------------------------------------
País: Poland
Capital: Warsaw
Región: Europe
Población: 37392000
----------------------------------------


La petición `GET` a REST Countries API se ha realizado correctamente y ha devuelto un código de estado `200`.

La respuesta recibida está en formato JSON y contiene información estructurada sobre países europeos. En la salida he mostrado algunos campos principales de los primeros países, como el nombre, la capital, la región y la población.

También he utilizado el parámetro `fields` para limitar la información recibida y trabajar solo con los datos necesarios para este ejercicio. En el caso del campo `capital`, como puede venir en forma de lista, lo he convertido en texto para conservar todas las capitales disponibles y no solo la primera.

### 2.4 Conversión de los datos a un DataFrame

En este apartado convierto los datos recibidos desde REST Countries API en un DataFrame de pandas.

Como la respuesta JSON contiene algunos campos anidados, preparo una lista nueva con los campos que quiero analizar de forma más clara.  
De esta manera, cada país queda representado como una fila y cada variable importante queda organizada como una columna.

Los campos que utilizo para construir el DataFrame son:

- Nombre del país.
- Capital o capitales.
- Región.
- Población.
- Idiomas oficiales.
- Monedas utilizadas.

In [10]:
countries_list = []

for country in countries_data:
    name = country.get("name", {}).get("common", "No disponible")
    
    capital_list = country.get("capital", [])
    capital = ", ".join(capital_list) if capital_list else "No disponible"
    
    region = country.get("region", "No disponible")
    population = country.get("population", "No disponible")
    
    languages_dict = country.get("languages", {})
    languages = ", ".join(languages_dict.values()) if languages_dict else "No disponible"
    
    currencies_dict = country.get("currencies", {})
    currencies = ", ".join(currencies_dict.keys()) if currencies_dict else "No disponible"
    
    countries_list.append({
        "country": name,
        "capital": capital,
        "region": region,
        "population": population,
        "languages": languages,
        "currencies": currencies
    })

df_countries = pd.DataFrame(countries_list)

df_countries.head()

,country,capital,region,population,languages,currencies
0,Slovenia,Ljubljana,Europe,2130638,Slovene,EUR
1,Sweden,Stockholm,Europe,10605098,Swedish,SEK
2,Switzerland,Bern,Europe,9082848,"French, Swiss German, Italian, Romansh",CHF
3,Guernsey,St. Peter Port,Europe,64781,"English, French, Guernésiais","GBP, GGP"
4,Poland,Warsaw,Europe,37392000,Polish,PLN


Los datos recibidos desde REST Countries API se han convertido correctamente en un DataFrame de pandas.

Antes de crear el DataFrame, he preparado algunos campos anidados como `name`, `capital`, `languages` y `currencies` para que la tabla final sea más clara y fácil de leer.

Esta transformación es importante porque las respuestas JSON de una API no siempre tienen una estructura plana. En muchos casos, como en este ejemplo, es necesario extraer y organizar los campos antes de analizarlos con pandas.

### Conclusión del Nivel 2

En este nivel he trabajado con REST Countries API, una API pública seleccionada desde el repositorio Public APIs.

He revisado sus endpoints principales, he realizado una petición `GET`, he comprobado el código de estado y he transformado la respuesta JSON en un DataFrame de pandas.

También he preparado algunos campos anidados para que la tabla final sea más clara y fácil de analizar.

## Nivel 3 - API de Open Data Barcelona

En este tercer nivel trabajo con la API de Open Data Barcelona para buscar y consultar datos públicos de la ciudad.

Open Data Barcelona utiliza una estructura basada en CKAN. Esto permite buscar datasets con `package_search`, revisar los detalles de un dataset concreto con `package_show` y consultar registros de un recurso mediante `datastore_search`.

En este apartado me interesa buscar un dataset relacionado con logística urbana, última milla, zonas de carga y descarga, tráfico, restricciones urbanas o sostenibilidad, ya que estos temas pueden tener relación con Kamport y con el proyecto final de Data Analytics.

El objetivo es localizar un dataset útil, revisar sus recursos disponibles, recuperar al menos 100 registros mediante la API y convertir los resultados en un DataFrame de pandas.

### 3.1 Búsqueda de datasets con package_search

En este apartado utilizo el endpoint `package_search` para buscar datasets dentro del portal de Open Data Barcelona.

La búsqueda se centra en temas relacionados con el proyecto final de Data Analytics y con Kamport: logística urbana, última milla, tráfico, zonas de carga y descarga, restricciones urbanas, calidad del aire y sostenibilidad.

En lugar de utilizar una palabra clave muy general, pruebo varias palabras más concretas para localizar datasets que puedan aportar valor a un análisis urbano aplicado a la logística sostenible en Barcelona.

In [16]:
open_data_base_url = "https://opendata-ajuntament.barcelona.cat/data/api/3/action"

search_url = f"{open_data_base_url}/package_search"

keywords = [
    "càrrega i descàrrega",
    "zones càrrega descàrrega",
    "trànsit",
    "vehicles",
    "vehicles propulsió",
    "vehicles servei",
    "carrers 30",
    "carrers vianants",
    "zona baixes emissions",
    "zbe",
    "qualitat aire",
    "contaminació atmosfèrica",
    "aparcament superfície",
    "aparcaments",
    "carril bici",
    "bicicleta"
]

all_datasets = []

for keyword in keywords:
    search_params = {
        "q": keyword,
        "rows": 8
    }
    
    response_search = requests.get(search_url, params=search_params)
    
    print(f"Palabra clave utilizada: {keyword}")
    print(f"Código de estado: {response_search.status_code}")
    
    if response_search.status_code == 200:
        search_data = response_search.json()
        datasets = search_data["result"]["results"]
        
        print(f"Cantidad de datasets devueltos en esta consulta: {len(datasets)}")
        
        for dataset in datasets:
            dataset_info = {
                "keyword": keyword,
                "name": dataset.get("name"),
                "title": dataset.get("title")
            }
            all_datasets.append(dataset_info)
            
            print(f"Nombre interno: {dataset.get('name')}")
            print(f"Título: {dataset.get('title')}")
            print("-" * 80)
    else:
        print("No se ha podido realizar la búsqueda para esta palabra clave.")
    
    print("=" * 80)

Palabra clave utilizada: càrrega i descàrrega
Código de estado: 200
Cantidad de datasets devueltos en esta consulta: 1
Nombre interno: zones-carrega-descarrega
Título: Loading and unloading areas in the city of Barcelona
--------------------------------------------------------------------------------
Palabra clave utilizada: zones càrrega descàrrega
Código de estado: 200
Cantidad de datasets devueltos en esta consulta: 1
Nombre interno: zones-carrega-descarrega
Título: Loading and unloading areas in the city of Barcelona
--------------------------------------------------------------------------------
Palabra clave utilizada: trànsit
Código de estado: 200
Cantidad de datasets devueltos en esta consulta: 8
Nombre interno: rasters-mapa-estrategic-soroll
Título: Raster noise maps from Strategic Noise Map of the city of Barcelona
--------------------------------------------------------------------------------
Nombre interno: denuncies_sancions_transit_bcn_tip_vehicle
Título: Vehicle types i

Después de revisar los resultados obtenidos con diferentes palabras clave relacionadas con Kamport y con el proyecto final de Data Analytics, he decidido continuar con el dataset `zones-carrega-descarrega`.

Este dataset contiene información sobre las zonas de carga y descarga de la ciudad de Barcelona. No contiene datos directos de entregas o paquetes, pero sí información sobre una infraestructura urbana muy relacionada con la logística de última milla.

Para un proyecto como Kamport, este tipo de datos puede ser útil para entender dónde existen espacios habilitados para operaciones de reparto, carga y descarga dentro de la ciudad.

Otros datasets encontrados, como la zona de bajas emisiones, la calidad del aire, los vehículos o los carriles bici, también pueden ser útiles para análisis futuros, pero este dataset tiene una relación más directa con la operativa logística urbana.

### 3.2 Consulta del dataset con package_show

Después de revisar los resultados de búsqueda con palabras clave relacionadas con el proyecto final y con Kamport, selecciono el dataset `zones-carrega-descarrega`.

Este dataset contiene información sobre las zonas de carga y descarga de la ciudad de Barcelona. Aunque no incluye datos directos de entregas o paquetes, sí describe una infraestructura urbana importante para la logística de última milla.

En este apartado utilizo el endpoint `package_show` para consultar los detalles del dataset seleccionado. El objetivo es revisar los recursos disponibles, sus formatos y comprobar si alguno de ellos se puede consultar mediante la API.

Para poder utilizar `datastore_search` en el siguiente paso, necesito identificar el `resource_id` del recurso adecuado.

In [14]:
selected_dataset = "zones-carrega-descarrega"

show_url = f"{open_data_base_url}/package_show"

show_params = {
    "id": selected_dataset
}

response_show = requests.get(show_url, params=show_params)

print(f"Código de estado: {response_show.status_code}")

if response_show.status_code == 200:
    dataset_details = response_show.json()["result"]
    
    print(f"Nombre del dataset: {dataset_details.get('name')}")
    print(f"Título: {dataset_details.get('title')}")
    print(f"Cantidad de recursos disponibles: {len(dataset_details.get('resources', []))}")
    print("\nRecursos disponibles:\n")
    
    for resource in dataset_details.get("resources", []):
        print(f"Nombre del recurso: {resource.get('name')}")
        print(f"Formato: {resource.get('format')}")
        print(f"Resource ID: {resource.get('id')}")
        print(f"Datastore active: {resource.get('datastore_active')}")
        print("-" * 80)
else:
    print("No se han podido obtener los detalles del dataset.")

Código de estado: 200
Nombre del dataset: zones-carrega-descarrega
Título: Loading and unloading areas in the city of Barcelona
Cantidad de recursos disponibles: 2

Recursos disponibles:

Nombre del recurso: opendatabcn_zones-carrega-i-descarrega-csv.csv
Formato: CSV
Resource ID: 2e4f19d4-1032-491c-9d42-fb35847b9dc6
Datastore active: True
--------------------------------------------------------------------------------
Nombre del recurso: opendatabcn_zones-carrega-i-descarrega.json
Formato: JSON
Resource ID: 14dc1afc-35d5-4834-ad87-693d1b3a4933
Datastore active: False
--------------------------------------------------------------------------------


Con `package_show` he obtenido los detalles del dataset `zones-carrega-descarrega`.

En la respuesta he revisado los recursos disponibles, su formato, su `resource_id` y si tienen `datastore_active`. Esta revisión es necesaria para elegir un recurso que pueda consultarse en el siguiente paso mediante `datastore_search`.

El `resource_id` será el valor clave para recuperar los registros reales del dataset.

### 3.3 Selección de un recurso CSV o JSON

En este apartado reviso los recursos disponibles del dataset `zones-carrega-descarrega` para seleccionar uno que pueda utilizar en la siguiente consulta.

El enunciado pide elegir un recurso disponible en formato CSV o JSON. Además, para poder consultar los registros mediante `datastore_search`, es importante comprobar que el recurso tenga `datastore_active` igual a `True`.

Por este motivo, filtro los recursos del dataset y muestro solo aquellos que tienen un formato adecuado para continuar con el ejercicio.

In [17]:
valid_resources = []

for resource in dataset_details.get("resources", []):
    resource_format = resource.get("format", "").upper()
    datastore_active = resource.get("datastore_active", False)
    
    if resource_format in ["CSV", "JSON"]:
        resource_info = {
            "name": resource.get("name"),
            "format": resource_format,
            "resource_id": resource.get("id"),
            "datastore_active": datastore_active
        }
        
        valid_resources.append(resource_info)

print(f"Cantidad de recursos CSV o JSON encontrados: {len(valid_resources)}")
print("\nRecursos válidos encontrados:\n")

for resource in valid_resources:
    print(f"Nombre del recurso: {resource['name']}")
    print(f"Formato: {resource['format']}")
    print(f"Resource ID: {resource['resource_id']}")
    print(f"Datastore active: {resource['datastore_active']}")
    print("-" * 80)

Cantidad de recursos CSV o JSON encontrados: 2

Recursos válidos encontrados:

Nombre del recurso: opendatabcn_zones-carrega-i-descarrega-csv.csv
Formato: CSV
Resource ID: 2e4f19d4-1032-491c-9d42-fb35847b9dc6
Datastore active: True
--------------------------------------------------------------------------------
Nombre del recurso: opendatabcn_zones-carrega-i-descarrega.json
Formato: JSON
Resource ID: 14dc1afc-35d5-4834-ad87-693d1b3a4933
Datastore active: False
--------------------------------------------------------------------------------


En esta revisión he filtrado los recursos del dataset para identificar aquellos que están disponibles en formato CSV o JSON.

También he comprobado el valor de `datastore_active`, porque para recuperar registros mediante `datastore_search` necesito un recurso que se pueda consultar desde la API.

El siguiente paso será seleccionar el `resource_id` adecuado y utilizarlo para recuperar al menos 100 registros del dataset.

### 3.4 Consulta de registros con datastore_search

En este apartado utilizo el endpoint `datastore_search` para recuperar registros reales del recurso seleccionado.

Para poder hacer esta consulta necesito un `resource_id`. En el paso anterior he revisado los recursos disponibles del dataset y ahora selecciono un recurso en formato CSV o JSON que tenga `datastore_active` igual a `True`.

El objetivo de esta consulta es recuperar al menos 100 registros del dataset para poder convertirlos posteriormente en un DataFrame de pandas.

In [18]:
active_resources = [
    resource for resource in valid_resources
    if resource["datastore_active"] == True
]

if active_resources:
    selected_resource = active_resources[0]
    
    resource_id = selected_resource["resource_id"]
    
    print("Recurso seleccionado:")
    print(f"Nombre: {selected_resource['name']}")
    print(f"Formato: {selected_resource['format']}")
    print(f"Resource ID: {resource_id}")
    print(f"Datastore active: {selected_resource['datastore_active']}")
else:
    print("No se ha encontrado ningún recurso activo para consultar con datastore_search.")

Recurso seleccionado:
Nombre: opendatabcn_zones-carrega-i-descarrega-csv.csv
Formato: CSV
Resource ID: 2e4f19d4-1032-491c-9d42-fb35847b9dc6
Datastore active: True


In [19]:
datastore_url = f"{open_data_base_url}/datastore_search"

datastore_params = {
    "resource_id": resource_id,
    "limit": 100
}

response_datastore = requests.get(datastore_url, params=datastore_params)

print(f"Código de estado: {response_datastore.status_code}")

if response_datastore.status_code == 200:
    datastore_data = response_datastore.json()
    records = datastore_data["result"]["records"]
    
    print(f"Cantidad de registros recuperados: {len(records)}")
    print("\nEjemplo del primer registro:\n")
    print(json.dumps(records[0], indent=4, ensure_ascii=False))
else:
    print("No se han podido recuperar los registros del recurso seleccionado.")

Código de estado: 200
Cantidad de registros recuperados: 100

Ejemplo del primer registro:

{
    "addresses_roadtype_name": "",
    "addresses_end_street_number": "",
    "values_attribute_name": "",
    "estimated_dates": "",
    "addresses_road_name": "C Mare de Déu de Port",
    "values_category": "",
    "addresses_zip_code": "8038",
    "secondary_filters_id": "",
    "values_value": "",
    "addresses_town": "BARCELONA",
    "geo_epgs_4326_y": "2.143753563396819",
    "geo_epgs_4326_x": "41.35737514116946",
    "secondary_filters_name": "",
    "secondary_filters_tree": "",
    "start_date": "",
    "addresses_district_name": "Sants-Montjuïc",
    "end_date": "",
    "geo_epgs_25831_x": "428378.3838331724",
    "addresses_start_street_number": "221",
    "register_id": "﻿99400601854",
    "institution_id": "",
    "addresses_main_address": "True",
    "addresses_district_id": "3",
    "addresses_roadtype_id": "",
    "addresses_type": "",
    "addresses_neighborhood_id": "13",
 

Con `datastore_search` he recuperado 100 registros del recurso seleccionado.

La consulta se ha realizado utilizando el `resource_id` del recurso activo identificado en el paso anterior. La respuesta recibida contiene los registros dentro de la clave `result["records"]`.

En el siguiente paso convertiré estos registros en un DataFrame de pandas para poder trabajar con ellos de forma tabular.

### 3.5 Conversión de los resultados a un DataFrame y exportación a CSV

En este apartado convierto los registros obtenidos con `datastore_search` en un DataFrame de pandas.

Esta transformación permite trabajar con los datos en formato tabular, revisar las columnas disponibles, comprobar las primeras filas y preparar la información para una posible limpieza o análisis posterior.

En este caso, cada registro recuperado desde la API se convierte en una fila del DataFrame.

Después de revisar la estructura básica del DataFrame, guardo los datos en un archivo `.csv`, tal como pide el enunciado del ejercicio.

In [23]:
df_loading_zones = pd.DataFrame(records)

print(f"Número de filas: {df_loading_zones.shape[0]}")
print(f"Número de columnas: {df_loading_zones.shape[1]}")

df_loading_zones.head()

Número de filas: 100
Número de columnas: 40


,addresses_roadtype_name,addresses_end_street_number,values_attribute_name,estimated_dates,addresses_road_name,values_category,addresses_zip_code,secondary_filters_id,values_value,addresses_town,...,geo_epgs_25831_y,institution_name,modified,secondary_filters_asia_id,secondary_filters_fullpath,values_description,values_id,addresses_neighborhood_name,values_outstanding,values_attribute_id
0,,,,,C Mare de Déu de Port,,8038,,,BARCELONA,...,4578783.95564586,,2022-09-17T10:29:14.130761,,,,,la Marina de Port,,
1,,,,,Av Mare de Déu de Montserrat,,8041,,,BARCELONA,...,4586225.225624279,,2022-09-17T10:29:13.456322,,,,,el Guinardó,,
2,,,,,C Amigó,,8021,,,BARCELONA,...,4583181.824702035,,2022-09-17T10:29:26.602724,,,,,Sant Gervasi - Galvany,,
3,,,,,C Sor Eulàlia d'Anzizu,,8034,,,BARCELONA,...,4582466.414148673,,2022-09-17T10:26:39.812146,,,,,Pedralbes,,
4,,,,,C Vila i Vilà,,8004,,,BARCELONA,...,4580707.149276413,,2022-09-17T10:26:42.144114,,,,,el Poble-sec,,


In [24]:
df_loading_zones.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 40 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   addresses_roadtype_name        100 non-null    str  
 1   addresses_end_street_number    100 non-null    str  
 2   values_attribute_name          100 non-null    str  
 3   estimated_dates                100 non-null    str  
 4   addresses_road_name            100 non-null    str  
 5   values_category                100 non-null    str  
 6   addresses_zip_code             100 non-null    str  
 7   secondary_filters_id           100 non-null    str  
 8   values_value                   100 non-null    str  
 9   addresses_town                 100 non-null    str  
 10  geo_epgs_4326_y                100 non-null    str  
 11  geo_epgs_4326_x                100 non-null    str  
 12  secondary_filters_name         100 non-null    str  
 13  secondary_filters_tree         1

Después de revisar la estructura del DataFrame con `info()`, puedo ver el número de columnas, el tipo de dato de cada una y si existen valores nulos.

Esta revisión es útil antes de exportar los datos, porque permite comprobar que la información se ha cargado correctamente desde la API y que el DataFrame está preparado para guardarse en un archivo `.csv`.

In [25]:
output_file = "zonas_carga_descarga_barcelona.csv"

df_loading_zones.to_csv(output_file, index=False)

print(f"Archivo guardado correctamente: {output_file}")

Archivo guardado correctamente: zonas_carga_descarga_barcelona.csv


Los registros recuperados desde la API se han convertido correctamente en un DataFrame de pandas.

El DataFrame permite revisar los datos en formato tabular, comprobar el número de filas y columnas y visualizar las primeras observaciones con `head()`.

Después, he guardado el DataFrame en un archivo `.csv` utilizando `index=False` para evitar que pandas añada una columna extra con el índice del DataFrame.

Esta conversión y exportación son importantes porque permiten reutilizar los datos en otros análisis o herramientas sin tener que volver a hacer la petición a la API.

## Conclusión final

En este sprint he practicado el consumo de datos desde diferentes APIs REST utilizando Python.

Primero he trabajado con JSONPlaceholder para entender los métodos HTTP principales y los códigos de estado. Después he utilizado REST Countries API para consultar datos reales en formato JSON y transformarlos en un DataFrame de pandas.

Finalmente, he trabajado con la API de Open Data Barcelona para buscar un dataset relacionado con logística urbana, recuperar 100 registros mediante `datastore_search`, convertirlos en un DataFrame y exportarlos a un archivo `.csv`.

Este flujo es útil para el trabajo de un Data Analyst, ya que muchas fuentes de datos reales se consultan mediante APIs y necesitan ser transformadas antes de poder analizarlas.